## Training and validation

**Goal:** Compare simple baselines and 2 pre-declared model pipelines on the 2020 validation period, then freeze one model family

In [7]:
#!pip install pandas numpy matplotlib missingno
#%pip install scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import missingno as msno
import joblib
import os
from pathlib import Path

#import sklearn.preprocessing

pd.options.display.max_rows=1000

import sys
import json
from IPython.display import display
import time

def find_project_root(start: Path = Path.cwd()) -> Path:
    """Find the parent directory containing src/data.py."""
    for candidate in (start, *start.parents):
        if (candidate / "src" / "data.py").is_file():
            return candidate

    raise FileNotFoundError("Could not find src/data.py")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
    


#from data import load_data, clean_prices, clean_stock_list, clean_financials
from config import ProjectConfig, ProjectPaths
from modeling import (
    build_model_pipeline
    ,chronological_split
    ,make_xy
    ,save_json
    ,select_recent_dates
    ,evaluate_predictions
)
from features import (
    infer_feature_columns
)


config = ProjectConfig()
paths = ProjectPaths(PROJECT_ROOT)
paths.ensure_output_dirs

print("Project root:", PROJECT_ROOT)
print("Imported load_data successfully")

print("Imported libraries sucessfully! yay")

Project root: c:\Users\MY NGOC\Documents\MAU_TACAS_68803\DA631E_ArtificialIntelligenceForDataScience\Project_1_SupervisedLearning
Imported load_data successfully
Imported libraries sucessfully! yay


In [8]:
FAST_MODE = True
MODEL_NAMES = ['ridge', 'hist_gradient_boosting']

In [9]:

features_table = pd.read_parquet(paths.processed / "features_table.parquet")
numeric_features, categorical_features = infer_feature_columns(features_table)
splits, split_report = chronological_split(features_table, config)
train = splits['train']
valid = splits['validation']

model_train = select_recent_dates(train, max_dates=config.fast_train_dates if FAST_MODE else None)

print(f"training rows used: {len(model_train):,} of {len(train)}")
print(f"training dates used: {model_train['Date'].nunique()}")

training rows used: 730,704 of 1395521
training dates used: 378


## Baseline

In [10]:
evaluations = {}
prediction_outputs = {}
ic_outputs = {}
spread_outputs = {}

baseline_predictions = {
    "baseline_zero": np.zeros(len(valid)),
    "baseline_momentum20": valid["Momentum20Days"].fillna(0.0).to_numpy(),
}

for name, predictions in baseline_predictions.items():
    metrics, ranked = evaluate_predictions(
        valid,
        predictions,
        config,
        strict_portfolio_size=True,
    )
    evaluations[name] = metrics
    prediction_outputs[name] = ranked


display(pd.DataFrame(evaluations).T)

,MAE,RMSE,R2
baseline_zero,0.019717,0.000842,-0.000112
baseline_momentum20,0.095870,0.020363,-23.173361


## Fit candidate pipelines

- **Ridge:** a stable linear benchmark with median imputation, missingness
  indicators, scaling, and one-hot categories.
- **Histogram gradient boosting:** a nonlinear model with median imputation
  and unknown-safe ordinal category encoding.

In [11]:
X_train, y_train = make_xy(model_train, numeric_features, categorical_features)
X_valid, y_valid = make_xy(valid, numeric_features, categorical_features)

fitted_models = {}
training_seconds = {}

for model_name in MODEL_NAMES:
    print(f"Fitting {model_name}...")
    pipeline = build_model_pipeline(
        model_name
        ,numeric_features
        ,categorical_features
    )
    
    started = time.perf_counter()
    pipeline.fit(X_train, y_train)
    training_seconds[model_name] = time.perf_counter() - started
    
    valid_predictions = pipeline.predict(X_valid)
    metrics, ranked = evaluate_predictions(
        valid
        ,valid_predictions
        ,config
    )
    metrics.loc["TrainingSeconds"] = training_seconds[model_name]
    
    fitted_models[model_name] = pipeline
    evaluations[model_name] = metrics
    prediction_outputs[model_name] = ranked
    joblib.dump(pipeline, paths.models / f"validation_{model_name}.joblib")
    print(f"Finished {model_name} in {training_seconds[model_name]:.1f} seconds")

Fitting ridge...
Finished ridge in 8.6 seconds
Fitting hist_gradient_boosting...
Finished hist_gradient_boosting in 11.3 seconds


## Compare

In [16]:
comparison = pd.DataFrame(evaluations).T.sort_values(by='R2',ascending=True)
display(comparison)



,MAE,R2,RMSE,TrainingSeconds
baseline_momentum20,0.095870,-23.173361,0.020363,NaN
hist_gradient_boosting,0.020027,-0.024327,0.000863,11.273969
ridge,0.019945,-0.010386,0.000851,8.553417
baseline_zero,0.019717,-0.000112,0.000842,NaN


In [15]:
evaluations

{'baseline_zero': MAE     0.019717
 RMSE    0.000842
 R2     -0.000112
 Name: value, dtype: float64,
 'baseline_momentum20': MAE      0.095870
 RMSE     0.020363
 R2     -23.173361
 Name: value, dtype: float64,
 'ridge': MAE                0.019945
 RMSE               0.000851
 R2                -0.010386
 TrainingSeconds    8.553417
 Name: value, dtype: float64,
 'hist_gradient_boosting': MAE                 0.020027
 RMSE                0.000863
 R2                 -0.024327
 TrainingSeconds    11.273969
 Name: value, dtype: float64}